In [ ]:
# Install the development version of braindecode
%pip install git+https://github.com/braindecode/braindecode.git@5bdd4ba2fb3608945f22814519ff640689bacafd


# Sleep staging on the Sleep Physionet dataset using U-Sleep network

<div class="alert alert-info"><h4>Note</h4>

Please take a look at the simpler sleep staging example
`sleep-staging-physionet-chambon2018`
before going through this example. The current example uses a more complex
architecture and a sequence-to-sequence (seq2seq) approach.

</div>

This tutorial shows how to train and test a sleep staging neural network with
Braindecode. We adapt the U-Sleep approach of [1]_ to learn on sequences of EEG
windows using the openly accessible Sleep Physionet dataset [2]_ [3]_.


In [ ]:
# Authors: Theo Gnassounou <theo.gnassounou@inria.fr>
#          Omar Chehab <l-emir-omar.chehab@inria.fr>
#
# License: BSD (3-clause)

## Loading and preprocessing the dataset

### Loading

First, we load the data using the
:class:`braindecode.datasets.sleep_physionet.SleepPhysionet` class. We load
two recordings from two different individuals: we will use the first one to
train our network and the second one to evaluate performance (as in the [MNE
sleep staging example](mne-clinical-60-sleep_)).




In [ ]:
from braindecode.datasets import SleepPhysionet

subject_ids = [0, 1]
dataset = SleepPhysionet(subject_ids=subject_ids, recording_ids=[2], crop_wake_mins=30)

### Preprocessing

Next, we preprocess the raw data. We scale each channel recording-wise to
have zero median and unit interquartile range. We don't upsample to 128 Hz as
done in [1]_ so that we keep the example as light as possible. No filtering
is described in [1]_.



In [ ]:
from sklearn.preprocessing import robust_scale

from braindecode.preprocessing import Preprocessor, preprocess

preprocessors = [Preprocessor(robust_scale, channel_wise=True)]

# Transform the data
preprocess(dataset, preprocessors)

### Extract windows

We extract 30-s windows to be used in the classification task.



In [ ]:
from braindecode.preprocessing import create_windows_from_events

mapping = {  # We merge stages 3 and 4 following AASM standards.
    "Sleep stage W": 0,
    "Sleep stage 1": 1,
    "Sleep stage 2": 2,
    "Sleep stage 3": 3,
    "Sleep stage 4": 3,
    "Sleep stage R": 4,
}

window_size_s = 30
sfreq = 100
window_size_samples = window_size_s * sfreq

windows_dataset = create_windows_from_events(
    dataset,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=0,
    window_size_samples=window_size_samples,
    window_stride_samples=window_size_samples,
    preload=True,
    mapping=mapping,
)

### Split dataset into train and valid

We split the dataset into training and validation set taking
every other subject as train or valid.



In [ ]:
split_ids = dict(train=subject_ids[::2], valid=subject_ids[1::2])
splits = windows_dataset.split(split_ids)
train_set, valid_set = splits["train"], splits["valid"]

## Create sequence samplers

Following the sequence-to-sequence approach of [1]_, we need to provide our
neural network with sequences of windows. We can achieve this by defining
Sampler objects that return sequences of windows.
Non-overlapping sequences of 35 windows are used in [1]_, however to limit
the memory requirements for this example we use shorter sequences of 3
windows.



In [ ]:
from braindecode.samplers import SequenceSampler

n_windows = 3  # Sequences of 3 consecutive windows; originally 35 in paper
n_windows_stride = 3  # Non-overlapping sequences

train_sampler = SequenceSampler(
    train_set.get_metadata(), n_windows, n_windows_stride, randomize=True
)
valid_sampler = SequenceSampler(valid_set.get_metadata(), n_windows, n_windows_stride)

# Print number of examples per class
print(len(train_sampler))
print(len(valid_sampler))

Finally, since some sleep stages appear a lot more often than others (e.g.
most of the night is spent in the N2 stage), the classes are imbalanced. To
avoid overfitting to the more frequent classes, we compute weights that we
will provide to the loss function when training.



In [ ]:
import numpy as np
from sklearn.utils import compute_class_weight

y_train = [train_set[idx][1][1] for idx in train_sampler]
class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)

## Create model

We can now create the deep learning model. In this tutorial, we use the
U-Sleep architecture introduced in [1]_, which is fully convolutional
neural network.



In [ ]:
import torch

from braindecode.models import USleep
from braindecode.util import set_random_seeds

cuda = torch.cuda.is_available()  # check if CUDA is available
mps = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
device = "cuda" if cuda else "mps" if mps else "cpu"
if cuda:
    torch.backends.cudnn.benchmark = True
# Set random seed to be able to roughly reproduce results
# Note that with cudnn benchmark set to True, GPU indeterminism
# may still make results substantially different between runs.
# To obtain more consistent results at the cost of increased computation time,
# you can set `cudnn_benchmark=False` in `set_random_seeds`
# or remove `torch.backends.cudnn.benchmark = True`
set_random_seeds(seed=31, cuda=cuda)

n_classes = 5
classes = list(range(n_classes))
# Extract number of channels and time steps from dataset
in_chans, input_size_samples = train_set[0][0].shape
model = USleep(
    n_chans=in_chans,
    sfreq=sfreq,
    depth=12,
    with_skip_connection=True,
    n_outputs=n_classes,
    n_times=input_size_samples,
)

# Send model to the selected accelerator
if device != "cpu":
    model.to(device)

## Training

We can now train our network. :class:`braindecode.classifier.EEGClassifier` is a
braindecode object that is responsible for managing the training of neural
networks. It inherits from :class:`skorch.classifier.NeuralNetClassifier`, so the
training logic is the same as in
[](skorch_).

<div class="alert alert-info"><h4>Note</h4>

We use different hyperparameters from [1]_, as these hyperparameters were
optimized on different datasets and with a different number of recordings.
Generally speaking, it is recommended to perform hyperparameter
optimization if reusing this code on a different dataset or with more
recordings.

</div>



In [ ]:
from skorch.callbacks import EarlyStopping, EpochScoring
from skorch.helper import predefined_split

from braindecode import EEGClassifier

lr = 1e-3
# Small batches give 46 parameter updates per epoch on the 375 training
# sequences of one night.
batch_size = 8
n_epochs = 20

from sklearn.metrics import balanced_accuracy_score


def balanced_accuracy_multi(model, X, y):
    y_pred = model.predict(X)
    return balanced_accuracy_score(y.flatten(), y_pred.flatten())


train_bal_acc = EpochScoring(
    scoring=balanced_accuracy_multi,
    on_train=True,
    name="train_bal_acc",
    lower_is_better=False,
)
valid_bal_acc = EpochScoring(
    scoring=balanced_accuracy_multi,
    on_train=False,
    name="valid_bal_acc",
    lower_is_better=False,
)
callbacks = [
    ("train_bal_acc", train_bal_acc),
    ("valid_bal_acc", valid_bal_acc),
    ("early_stopping", EarlyStopping(patience=10, load_best=True)),
]

clf = EEGClassifier(
    model,
    criterion=torch.nn.CrossEntropyLoss,
    criterion__weight=torch.Tensor(class_weights).to(device),
    optimizer=torch.optim.Adam,
    iterator_train__shuffle=False,
    iterator_train__sampler=train_sampler,
    iterator_valid__sampler=valid_sampler,
    train_split=predefined_split(valid_set),  # using valid_set for validation
    optimizer__lr=lr,
    batch_size=batch_size,
    callbacks=callbacks,
    device=device,
    classes=classes,
)
# Deactivate the default valid_acc callback:
clf.set_params(callbacks__valid_acc=None)

# Model training for a specified number of epochs. `y` is None as it is
# already supplied in the dataset.
clf.fit(train_set, y=None, epochs=n_epochs)

### Plot training curves

Panel A shows the weighted cross-entropy loss and panel B the balanced
accuracy on the training (grey) and validation (blue) recordings. With five
sleep stages, a classifier that guesses at random reaches a balanced
accuracy of 20 %, drawn as the dashed line. The validation balanced accuracy
rises from 41.5 % after the first epoch to 70.0 % at epoch 18, the epoch
with the lowest validation loss, whose weights early stopping restores at the
end of training.



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import MaxNLocator

BLUE, GREY, INK = "#0072B2", "#7F7F7F", "#14181F"
# MOABB, imported by ``braindecode.datasets``, sets a seaborn style when it is
# imported, so we go back to the matplotlib defaults before our own settings.
plt.rcdefaults()
plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10.5,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "legend.frameon": False,
    }
)

df = pd.DataFrame(clf.history.to_list())
fig, (ax_loss, ax_acc) = plt.subplots(2, 1, figsize=(6.8, 5.2), sharex=True)
for ax, metric in ((ax_loss, "loss"), (ax_acc, "bal_acc")):
    for split, name, color in (("train", "Train", GREY), ("valid", "Valid", BLUE)):
        values = df[f"{split}_{metric}"]
        ax.plot(df["epoch"], values, color=color, linewidth=1.8)
        ax.annotate(
            name,
            xy=(df["epoch"].iloc[-1], values.iloc[-1]),
            xytext=(6, 0),
            textcoords="offset points",
            color=color,
            fontweight="bold",
            va="center",
        )
    ax.grid(axis="y", color="#EAEAEA", linewidth=0.7)
    ax.set_axisbelow(True)
ax_acc.axhline(1 / n_classes, color=GREY, linestyle="--", linewidth=0.9)
ax_acc.text(
    df["epoch"].iloc[-1],
    1 / n_classes - 0.02,
    f"Chance ({100 / n_classes:.0f}%)",
    color=GREY,
    fontsize=8.5,
    ha="right",
    va="top",
)
ax_acc.set_ylim(0, 1)
ax_acc.set_yticks([0, 0.5, 1])
ax_loss.set_ylabel("Loss")
ax_acc.set_ylabel("Balanced accuracy")
ax_acc.set_xlabel("Epoch")
ax_acc.xaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
ax_loss.set_title("A  Cross-entropy loss", loc="left", fontweight="bold", color=INK)
ax_acc.set_title("B  Balanced accuracy", loc="left", fontweight="bold", color=INK)
fig.tight_layout()
plt.show()

Finally, we also display the confusion matrix and classification report.
Each row is normalized by the number of expert-labelled windows of that
stage, so the diagonal reads as the recall of each stage. N1, the rarest
stage in the training night, is the hardest: most of its windows are
predicted as REM.




In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

stage_names = ["Wake", "N1", "N2", "N3", "REM"]  # label order of ``mapping``
# U-Sleep predicts one stage per window, so we flatten the sequences.
y_true = np.array([valid_set[i][1] for i in valid_sampler]).flatten()
y_pred = clf.predict(valid_set).flatten()

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3, 4],
    display_labels=stage_names,
    normalize="true",
    cmap="Blues",
    values_format=".2f",
    colorbar=False,
    ax=ax,
)
ax.spines[["left", "bottom"]].set_visible(False)
ax.set_xlabel("Predicted stage")
ax.set_ylabel("Expert stage")
fig.tight_layout()
plt.show()

print(classification_report(y_true, y_pred))

Finally, we can also visualize the hypnogram of the recording we used for
validation: the expert annotations (A) and the predicted sleep stages (B),
drawn on the same time axis with the clinical stage order (Wake at the top,
N3 at the bottom). Each step is one 30-s window.
The predicted hypnogram follows the main sleep cycles of the expert one.



In [ ]:
# sphinx_gallery_thumbnail_number = 3

# Row of each label in the hypnogram: Wake at the top, N3 at the bottom.
stage_height = {0: 4, 4: 3, 1: 2, 2: 1, 3: 0}
step_h = window_size_s / 3600
hours = np.arange(len(y_true) + 1) * step_h

fig, axes = plt.subplots(2, 1, figsize=(7.5, 4.2), sharex=True, sharey=True)
for ax, labels, name, color, letter in (
    (axes[0], y_true, "Expert", GREY, "A"),
    (axes[1], y_pred.flatten(), "Predicted", BLUE, "B"),
):
    heights = [stage_height[int(label)] for label in labels]
    ax.step(hours, heights + heights[-1:], where="post", color=color, linewidth=1.2)
    ax.set_yticks([4, 3, 2, 1, 0])
    ax.set_yticklabels(["Wake", "REM", "N1", "N2", "N3"])
    ax.set_ylim(-0.4, 4.4)
    ax.set_title(f"{letter}  {name}", loc="left", fontweight="bold", color=INK)
axes[1].set_xlim(0, hours[-1])
axes[1].set_xlabel("Time (h)")
fig.tight_layout()
plt.show()

## References

.. [1] Perslev M, Darkner S, Kempfner L, Nikolic M, Jennum PJ, Igel C.
       U-Sleep: resilient high-frequency sleep staging. npj Digit. Med. 4, 72 (2021).
       https://github.com/perslev/U-Time/blob/master/utime/models/usleep.py

.. [2] B Kemp, AH Zwinderman, B Tuk, HAC Kamphuisen, JJL Oberyé. Analysis of
       a sleep-dependent neuronal feedback loop: the slow-wave
       microcontinuity of the EEG. IEEE-BME 47(9):1185-1194 (2000).

.. [3] Goldberger AL, Amaral LAN, Glass L, Hausdorff JM, Ivanov PCh,
       Mark RG, Mietus JE, Moody GB, Peng C-K, Stanley HE. (2000)
       PhysioBank, PhysioToolkit, and PhysioNet: Components of a New
       Research Resource for Complex Physiologic Signals.
       Circulation 101(23):e215-e220

.. include:: /links.inc

